# Imaging physics: coherence, focus, polarization, immersion, bandwidth

This notebook uses the engine to demonstrate five pieces of projection-imaging
physics, each checked against a closed form:

1. **partial coherence** sets the resolution limit $\lambda / (\mathrm{NA} (1 + \sigma))$;
2. **off-axis illumination** extends the depth of focus (DOF), and only an exact
   defocus model shows it;
3. **vector imaging**: TE keeps its contrast at high NA, TM loses it;
4. **immersion** prints pitches that no dry lens can;
5. **broadband sources**: per-wavelength imaging of an HHG comb;

plus a short look at EUV at NA 0.33 vs High-NA 0.55.

Model status: scalar Hopkins imaging, exact defocus, vector imaging, immersion
and per-wavelength imaging are ✅; EUV projection optics are 🔶 (isotropic pupil,
no anamorphic magnification or mask 3D). See the
[capability matrix](../../docs/capability-matrix.md).

Where a curve is sensitive to how the source pupil is sampled (near-coherent
fills, pitches next to a cutoff), the engine is given an explicit fine source
grid with `source_points_per_axis`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import highuvlith as huv
from highuvlith import viz

%matplotlib inline

def line_space_engine(source, optics, cd, pitch, *, px_per_pitch=64, spp=41, **kw):
    """Engine for a periodic L/S on a one-pitch commensurate grid."""
    mask = huv.MaskConfig.line_space(cd, pitch)
    grid = mask.commensurate_grid(size=px_per_pitch, target_pixel_nm=pitch / px_per_pitch)
    return huv.SimulationEngine(source, optics, mask, grid=grid, max_kernels=40,
                                source_points_per_axis=spp, **kw)

## 1. Partial coherence and the resolution limit

A grating of pitch $p$ diffracts into orders at pupil positions $\pm \lambda / p$
(in units of NA/λ, i.e. at $\pm \lambda / (p \mathrm{NA})$). An image forms only if at
least the zero and one first order pass the pupil together. For a coherent
on-axis source that requires $p > \lambda / \mathrm{NA}$; a source of radius σ tilts
part of the illumination by up to σ, so the limit becomes

$p_{min} = \lambda / (\mathrm{NA} (1 + \sigma))$.

Below: contrast vs pitch at 157.63 nm, NA 0.75, for four fills, with the
predicted cutoffs as dashed lines. Larger σ resolves smaller pitches, at the price
of lower contrast at large pitches.

In [ ]:
wl, na = 157.63, 0.75
optics = huv.OpticsConfig(numerical_aperture=na)
pitches = np.arange(110.0, 330.0, 5.0)
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, sigma in enumerate((0.1, 0.3, 0.5, 0.8)):
    src = huv.SourceConfig.f2_laser(sigma=sigma)
    c = [line_space_engine(src, optics, p / 2, p).compute_aerial_image().image_contrast() for p in pitches]
    line, = ax.plot(pitches, c, label=f"σ = {sigma}")
    ax.axvline(wl / (na * (1 + sigma)), color=line.get_color(), ls="--", lw=1)
ax.axvline(wl / na, color="grey", ls=":", lw=1)
ax.text(wl / na + 2, 0.05, "coherent limit λ/NA", color="grey", fontsize=9)
ax.set(xlabel="pitch (nm), 1:1 lines and spaces", ylabel="image contrast", ylim=(0, 1.02),
       title="Resolution limit vs partial coherence (157.63 nm, NA 0.75)")
ax.legend()
ax.grid(alpha=0.3)

## 2. Off-axis illumination and depth of focus

### Why off-axis light helps

Defocus adds the phase $\phi(\rho) = (2 \pi n / \lambda) z (1 - \sqrt{1 - (\mathrm{NA} \rho / n)^2})$
to a ray at pupil radius ρ. With a conventional fill, the zero and first orders of
a dense grating travel at different pupil radii, pick up different phases and
drift out of step as $z$ grows. If instead the illumination comes from two poles
placed at $\sigma_c = \lambda / (2 p \mathrm{NA})$, the zero order of one pole and the
first order of the same pole sit symmetrically at $\pm \sigma_c$: same radius,
same defocus phase, so the two-beam fringe barely changes with focus.

### Why the defocus model matters

The engine evaluates that phase **inside the pupil, for every source point**
(`defocus_model="exact"`, the default). An older shortcut,
`defocus_model="kernel_phase"`, applied one paraxial phase per mask frequency
after the kernels were built. That is exact only for an on-axis coherent source.
For any off-axis light it attaches the wrong phase to each order: the
dipole's DOF advantage disappears and the image refocuses periodically with
defocus, an artifact. The 90 nm lines on a 180 nm pitch below are at
$k_1 \approx 0.43$.

In [ ]:
cd, pitch = 90.0, 180.0
sigma_c = wl / (2 * pitch * na)
fills = {
    "conventional σ 0.7": huv.SourceConfig.f2_laser(sigma=0.7),
    "annular 0.5-0.8": huv.SourceConfig.f2_laser().with_illumination("annular", 0.5, 0.8),
    f"dipole σc {sigma_c:.2f}": huv.SourceConfig.f2_laser().with_illumination("dipole", sigma_c, 0.1, 0.0),
}
focuses = np.linspace(-400.0, 400.0, 33)
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4.3), sharey=True)
for name, src in fills.items():
    for ax, model in ((ax0, "exact"), (ax1, "kernel_phase")):
        eng = line_space_engine(src, optics, cd, pitch, defocus_model=model)
        c = [p.image_contrast() for p in eng.compute_through_focus(list(focuses))]
        ax.plot(focuses, c, label=name)
ax0.set(title="exact defocus (default)", xlabel="defocus (nm)", ylabel="image contrast", ylim=(0, 1))
ax1.set(title="legacy kernel_phase (wrong off axis)", xlabel="defocus (nm)")
for ax in (ax0, ax1):
    ax.grid(alpha=0.3)
ax0.legend()
fig.tight_layout()

`compare_illumination` runs the same comparison in one call (exact defocus) and
also reports CD and NILS at a 0.3 threshold. Note that the quadrupole, built for
two-dimensional patterns, does worse than the dipole on a one-dimensional
grating.

In [ ]:
for row in huv.compare_illumination(cd, pitch, focuses_nm=[-200.0, 0.0, 200.0], grid_size=64, pixel_nm=3.0):
    print(f"{row.name:24s} contrast {row.contrast:.3f}   at ±200 nm {row.contrast_through_focus[0]:.3f}   "
          f"NILS {row.nils:.2f}")

## 3. Vector imaging: TE and TM at high NA

Scalar imaging adds the diffraction orders as if light had no polarization. In
reality two orders crossing at a half-angle θ in the image space interfere with
their electric fields: **TE** (field parallel to the lines) fields are parallel
and interfere fully, **TM** fields are tilted by $2 \theta$ and interfere with a
factor $\cos 2 \theta$. For two beams at the edge of a pupil of numerical aperture
NA the contrast is

- TE: 1;  TM: $\lvert 1 - 2 \mathrm{NA}^2 \rvert$ (zero at NA $\approx 0.71$, reversed above);
- unpolarized: $1 - \mathrm{NA}^2$.

`viz.plot_polarization_contrast` draws these from the engine's vector pupil
(`vector_two_beam_contrast`).

In [ ]:
ax = viz.plot_polarization_contrast()
print("NA 0.95:", {pol: round(huv.vector_two_beam_contrast(0.95, pol), 3) for pol in ("y", "x", "unpolarized")})

Through the full engine: 45 nm lines at 90 nm pitch, ArF water immersion (NA
1.35), x-dipole. The two interfering orders meet in water at
$\sin \theta = \lambda / (2 p n) \approx 0.75$, so TM keeps only about
$\lvert \cos 2 \theta \rvert \approx 0.12$ of the TE modulation, and unpolarized light
averages the two. Polarized illumination is therefore mandatory for
hyper-NA immersion tools.

In [ ]:
common = dict(wavelength_nm=193.368, na=1.35, immersion=True,
              illumination=("dipole", 0.8, 0.1, 0.0), grid_size=64, pixel_nm=1.5)
for pol in ("te", "tm", "unpolarized"):
    r = huv.simulate_line_space(45.0, 90.0, imaging="vector", polarization=pol, **common)
    print(f"vector {pol:12s} contrast {r.contrast:.3f}")
print(f"scalar              contrast {huv.simulate_line_space(45.0, 90.0, **common).contrast:.3f}")

## 4. Immersion

A dry lens cannot exceed NA 1. Filling the gap between the last lens and the
wafer with water (n = 1.437 at 193 nm) lets the lens deliver
$\mathrm{NA} = n \sin \theta$ up to about 1.35 in production. The minimum pitch
$\lambda / (\mathrm{NA} (1 + \sigma))$ shrinks accordingly; the defocus phase is
evaluated in the medium. Below: contrast vs pitch for an annular 0.7-0.9 fill at
193.368 nm, dry NA 0.93 vs water NA 1.35.

In [ ]:
arf = huv.SourceConfig.arf_laser().with_illumination("annular", 0.7, 0.9)
pitches_193 = np.arange(70.0, 160.0, 4.0)
fig, ax = plt.subplots(figsize=(8, 4.2))
for label, opt in (("dry NA 0.93", huv.OpticsConfig(numerical_aperture=0.93)),
                   ("water NA 1.35 (193i)", huv.OpticsConfig.immersion_193i())):
    c = [line_space_engine(arf, opt, p / 2, p).compute_aerial_image().image_contrast() for p in pitches_193]
    line, = ax.plot(pitches_193, c, label=label)
    ax.axvline(193.368 / (opt.numerical_aperture * 1.9), color=line.get_color(), ls="--", lw=1)
ax.axvline(90.0, color="grey", ls=":", lw=1)
ax.text(91, 0.9, "90 nm pitch", color="grey", fontsize=9)
ax.set(xlabel="pitch (nm)", ylabel="image contrast", ylim=(0, 1),
       title="ArF dry vs water immersion, annular 0.7-0.9 (dashed: cutoffs)")
ax.legend()
ax.grid(alpha=0.3)

## 5. Broadband sources: per-wavelength imaging of an HHG comb

Most lithography sources are narrow-band, and the engine's
`compute_polychromatic` treats the bandwidth as a small chromatic focus shift
around the centre wavelength (🔶, valid for $\Delta\lambda / \lambda \ll 1$). A
high-harmonic comb is not narrow-band: it is a set of discrete lines several
percent apart. `compute_multiwavelength` rebuilds the TCC at every spectral
sample and sums the images incoherently (✅).

Here the full neon comb, filtered to 12.5-20 nm, images 1:1 gratings through an
NA 0.33 pupil. Each harmonic has its own cutoff pitch
$\lambda_q / (\mathrm{NA} (1 + \sigma))$: the short harmonics still resolve pitches
that the centre wavelength cannot, and the long ones wash out pitches the centre
wavelength resolves. The monochromatic and narrow-band models miss both effects.

In [ ]:
comb = huv.SourceConfig.hhg(full_comb=True, comb_passband_nm=(12.5, 20.0))
lines = sorted({round(w, 2) for w, weight in comb.spectrum() if weight > 0.05})
print(f"centre {comb.wavelength_nm:.2f} nm, harmonic lines (nm): {lines}")
euv = huv.OpticsConfig.euv_nxe()
pitches_euv = np.arange(34.0, 72.0, 2.0)
curves = {"monochromatic (centre λ)": [], "narrow-band (polychromatic)": [], "per-wavelength (multiwavelength)": []}
for p in pitches_euv:
    eng = line_space_engine(comb, euv, p / 2, p, spp=61)
    curves["monochromatic (centre λ)"].append(eng.compute_aerial_image().image_contrast())
    curves["narrow-band (polychromatic)"].append(eng.compute_polychromatic().image_contrast())
    curves["per-wavelength (multiwavelength)"].append(eng.compute_multiwavelength().image_contrast())
fig, ax = plt.subplots(figsize=(8, 4.2))
for label, c in curves.items():
    ax.plot(pitches_euv, c, "o-", ms=3, label=label)
ax.set(xlabel="pitch (nm)", ylabel="image contrast", ylim=(0, 1.02),
       title="HHG comb 12.5-20 nm through NA 0.33")
ax.legend()
ax.grid(alpha=0.3)

## 6. EUV: NA 0.33 vs High-NA 0.55

At 13.5 nm the same scaling holds. An 18 nm pitch (9 nm lines) is below the
NA 0.33 cutoff even with a σ 0.9 fill, $13.5 / (0.33 \cdot 1.9) \approx 21.5$ nm, but
within reach of the 0.55 NA pupil. The High-NA preset assumes a central
obscuration of 0.2 NA (an assumption, not a published figure) and does not model
the anamorphic 4×/8× magnification or mask 3D effects (🔶).

In [ ]:
lpp = huv.SourceConfig.lpp_sn_13nm5()
for label, opt in (("NXE NA 0.33", huv.OpticsConfig.euv_nxe()), ("High-NA 0.55", huv.OpticsConfig.euv_high_na())):
    for p in (18.0, 24.0, 32.0):
        c = line_space_engine(lpp, opt, p / 2, p).compute_aerial_image().image_contrast()
        print(f"{label:13s} pitch {p:4.0f} nm  contrast {c:.3f}")

Docs: [Imaging pipeline](../../docs/pipeline.md), [Optics](../../docs/optics.md),
[Vector imaging](../../docs/vector-imaging.md),
[Masks and metrics](../../docs/masks-and-metrics.md).